# 02 — Compute capacity: CPU-hour and GPU-hour balance

Is there a *volume* shortfall? Demand is computed per workload as cores × wall time × jobs per year,
attributed to the cluster the group named ("Both" splits 50:50; "Not decided"/"Don't know" go to *unattributed*).
Capacity = cores × 8,760 h × the committee's target utilisation. **Deficit = demand − capacity** (positive = shortfall).
*Used* = last 12 months; *desired* = what groups would run if limits did not bind (an upper bound).

*Principles: P2, P7, P9, P12 in the [analysis framework](https://github.com/svaradh/hpc-questionnaire/blob/main/docs/analysis-framework.md). Nothing here scores or ranks groups; flags are candidates for the committee to look at. Totals cover responding groups only, and "Don't know" is never counted as zero; a yes/no flag shown as `<NA>` means its inputs are unknown. Only the respondents' answers and the cluster configuration are used — nothing depends on which code a workload runs.*

**To run:** paste the Spreadsheet ID in the first code cell, then *Runtime → Run all*. Colab asks once per session to sign in with your IISER Google account.

In [ ]:
SPREADSHEET_ID = 'PASTE_YOUR_SPREADSHEET_ID_HERE'   # <- the long string in the sheet URL
REPO_REF = 'main'                                   # git branch/tag for the helper code and cluster config

import os, urllib.request
if not os.path.exists('hpc_analysis.py'):            # in Colab: fetch the helper modules once per session
    for f in ['sheets_client.py', 'hpc_analysis.py', 'hpc_plots.py', 'code_capabilities.json']:
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/svaradh/hpc-questionnaire/{REPO_REF}/analysis/{f}', f)

import pandas as pd
from sheets_client import load_data, load_config
import hpc_analysis as ha
import hpc_plots as hp
pd.set_option('display.max_colwidth', 120)

dfs = load_data(SPREADSHEET_ID)
cfg = load_config(ref=REPO_REF)
wl = ha.workloads(dfs, cfg)
cov = ha.coverage(dfs)
print(f"{cov['groups']} groups responded; {cov['workloads']} workloads described "
      f"({cov['groups_with_workloads']} groups with at least one).")

## CPU core-hours per year

In [ ]:
cpu = ha.cpu_balance(wl, cfg)
hp.balance(cpu, 'CPU core-hours per year: demand vs deliverable capacity',
           {'used': 'Used (12 months)', 'desired': 'Desired (unconstrained)'}, ylabel='Million core-hours', scale=1e6)
print(ha.deficit_summary(cpu, 'used', 'capacity', ' core-h', 'used'))
print(ha.deficit_summary(cpu, 'desired', 'capacity', ' core-h', 'desired'))
print(f"Workloads with unknown inputs (not in the totals): {cpu.attrs['workloads_with_unknown_inputs']}")
cpu.round(0)

Largest contributors to desired CPU demand (to see whether a deficit is broad or concentrated):

In [ ]:
wl[wl['device'] != 'gpu'].nlargest(10, 'cpu_h_desired')[['pi_name', 'entry', 'cluster', 'cores', 'walltime_h', 'jobs_year', 'jobs_year_unconstrained', 'cpu_h_used', 'cpu_h_desired']]

## GPU-hours per year

GPU capacity stays *n/a* until the Raman GPU node's GPU count is entered in `clusters.json`; demand is shown regardless.

In [ ]:
gpu = ha.gpu_balance(wl, cfg)
hp.balance(gpu, 'GPU-hours per year: demand vs capacity', {'used': 'Used (12 months)', 'desired': 'Desired (unconstrained)'}, ylabel='GPU-hours')
print(ha.deficit_summary(gpu, 'desired', 'capacity', ' GPU-h', 'desired'))
gpu

## GPU:CPU balance

- **Latent GPU demand**: CPU workloads whose code has a *tested* GPU version. If moved to GPUs they would need roughly the GPU node-hours shown (a range from the reported speed-up bin), freeing their CPU core-hours.
- **CPU viability of GPU workloads**: whether GPU work could fall back to CPU nodes.
- **GPU versions of CPU codes**: how much CPU work could become GPU work.

In [ ]:
display(ha.gpu_latent_demand(wl, cfg))
hp.counts(wl.loc[wl['device'] == 'gpu', 'cpu_viable'].replace('', 'not answered'), 'GPU workloads: could they run on CPU-only nodes?')
hp.counts(wl.loc[wl['device'] != 'gpu', 'gpu_version'].replace('', 'not answered'), 'CPU workloads: does a GPU version of the code exist?')